# Lab 01 — Autonomous Bug-Fix Agent
### Fix It Fast: Autonomous Bug Resolution with Claude Code on Amazon Bedrock AgentCore

In this lab you will watch Claude Code autonomously:
1. **Read** a real GitHub issue
2. **Fetch and analyse** the buggy source code
3. **Fix** the bug and commit the change
4. **Open a pull request** for human review

**Architecture**

```
Your machine
  └─ connect.py ──(SigV4 WebSocket)──► AgentCore Runtime (microVM)
                                            └─ Claude Code
                                                  ├─ Amazon Bedrock  (model inference)
                                                  └─ AgentCore Gateway  (GitHub MCP tools)
                                                        └─ GitHub API
```

---
### Prerequisites

| Requirement | Check |
|---|---|
| AWS credentials | `aws sts get-caller-identity` |
| Python 3.10+ | `python3 --version` |
| `gh` CLI authenticated | `gh auth status` |
| `lab-00-deploy.ipynb` completed (or instructor pre-deployed) | `runtime_config.json` exists |

## Step 1 — Configuration

In [ ]:
import os, subprocess, json, sys, re

WORKSHOP_DIR = os.path.abspath("")
AGENT        = "claude-code"   # ← swap to "codex" or another agent folder name

# ── Load config written by lab-00 (override any value below if needed) ────────
_cfg_path = os.path.join(WORKSHOP_DIR, "workshop_config.json")
if os.path.exists(_cfg_path):
    _cfg = json.load(open(_cfg_path))
    GITHUB_USER = _cfg["GITHUB_USER"]
    GITHUB_REPO = _cfg["GITHUB_REPO"]
    AWS_REGION  = _cfg["AWS_REGION"]
    AWS_PROFILE = _cfg["AWS_PROFILE"]
    print("Loaded config from workshop_config.json")
else:
    GITHUB_USER = "your-github-username"
    GITHUB_REPO = "my-task-manager"
    AWS_REGION  = "us-west-2"
    AWS_PROFILE = ""  # set to your AWS profile name, or leave empty to use default
    print("workshop_config.json not found — using defaults above")
# ─────────────────────────────────────────────────────────────────────────────

os.environ["AWS_REGION"]         = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
os.environ["AWS_PROFILE"]        = AWS_PROFILE
os.environ["NO_COLOR"]           = "1"

AGENT_DIR = os.path.join(WORKSHOP_DIR, "coding_agents", AGENT)

_ANSI = re.compile(r"\x1b\[[0-9;]*m")

def run(cmd, cwd=None, capture=False):
    env = os.environ.copy()
    if capture:
        r = subprocess.run(cmd, shell=True, cwd=cwd, env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
        r.check_returncode()
        return _ANSI.sub("", r.stdout).strip()
    subprocess.run(cmd, shell=True, cwd=cwd, env=env, check=True)

print(f"Region:  {AWS_REGION}")
print(f"Profile: {AWS_PROFILE}")
print(f"Repo:    {GITHUB_USER}/{GITHUB_REPO}")
print(f"Agent:   {AGENT}")

## Step 2 — Verify credentials and runtime

In [ ]:
identity = json.loads(run("aws sts get-caller-identity --output json", capture=True))
print(f"AWS Account: {identity['Account']}")
print(f"Identity:    {identity['Arn']}")

with open(os.path.join(AGENT_DIR, "runtime_config.json")) as f:
    RUNTIME_ARN = json.load(f)["runtime_arn"]

print(f"\nRuntime ARN: {RUNTIME_ARN}")
print("\nReady.")

## Step 3 — Install dependencies

In [ ]:
import sys
run(f"{sys.executable} -m pip install -q -r {os.path.join(WORKSHOP_DIR, 'coding_agents', 'requirements.txt')}")
print("Done.")

## Step 4 — Browse the open bug issues

The repository has 5 bugs labelled by the triage agent. Read through them and pick one to fix.

In [ ]:
issues_raw = run(
    f"gh issue list --repo {GITHUB_USER}/{GITHUB_REPO} --state open "
    f"--label bug --json number,title,body,labels --limit 20",
    capture=True
)
issues = json.loads(issues_raw)

if not issues:
    print("No open issues labelled 'bug' yet — run lab-01 triage agent first.")
else:
    print(f"Open bug issues in {GITHUB_USER}/{GITHUB_REPO}:\n")
    for i in issues:
        print(f"  #{i['number']:2d}  {i['title']}")
        if i.get("body"):
            print(f"       {i['body'].splitlines()[0]}")
        print()
    print(f"Total: {len(issues)} bug issues")

## Step 5 — Pick an issue and run the agent

Change `TARGET_ISSUE` to any number from the list above, then run the next cell.

The agent will:
- Read the issue body via GitHub MCP tools
- Read `tests/test_bugs.py` to find the failing test for this bug
- Fetch the relevant source file and apply the fix
- Run `pytest` inside the microVM — iterate until the test passes
- Open a pull request with pytest output as evidence

**Watch the output — you'll see every tool call and test run in real time.**

In [ ]:
TARGET_ISSUE = issues[0]["number"]  # ← change this to any issue number

title = next(i["title"] for i in issues if i["number"] == TARGET_ISSUE)
print(f"Fixing issue #{TARGET_ISSUE}: {title}\n")

In [ ]:
PROMPT = (
    f"Fix GitHub issue #{TARGET_ISSUE} in the repository {GITHUB_USER}/{GITHUB_REPO}. "
    f"Read the issue, locate the bug in the code, apply the fix, and open a pull request."
)

print(f"Prompt: {PROMPT}")
print("=" * 60)

run(f'{sys.executable} connect.py --prompt "{PROMPT}"', cwd=AGENT_DIR)

## Step 6 — Review the pull request

In [ ]:
prs = json.loads(run(
    f"gh pr list --repo {GITHUB_USER}/{GITHUB_REPO} --state open "
    f"--json number,title,headRefName,url --limit 10",
    capture=True
))

if not prs:
    print("No open PRs yet — did the agent finish? Check the output above.")
else:
    for pr in prs:
        print(f"PR #{pr['number']}: {pr['title']}")
        print(f"  Branch: {pr['headRefName']}")
        print(f"  URL:    {pr['url']}")
        print()

In [ ]:
# Show the diff — does the fix look correct?
if prs:
    print(f"Diff for PR #{prs[0]['number']}:\n")
    run(f"gh pr diff {prs[0]['number']} --repo {GITHUB_USER}/{GITHUB_REPO}")

## Step 7 — Explore how it works

Two files control the agent's behaviour. Read them to understand the design.

**`CLAUDE.md`** — the system prompt baked into the container. Sets the agent's role, which tools to use, and guardrails (never merge, never close issues, always label).

**`run.sh`** — the container launch script. Reads `GATEWAY_URL` from the runtime environment and writes it into `~/.mcp.json` so Claude Code auto-connects to the GitHub MCP tools on startup.

In [ ]:
print("=" * 60)
print("CLAUDE.md  (system prompt baked into the container)")
print("=" * 60)
with open(os.path.join(AGENT_DIR, "CLAUDE.md")) as f:
    print(f.read())

In [ ]:
print("=" * 60)
print("run.sh  (container launch script)")
print("=" * 60)
with open(os.path.join(AGENT_DIR, "run.sh")) as f:
    print(f.read())

## Step 8 — Try another issue

Go back to Step 5, change `TARGET_ISSUE`, and re-run. The 5 bugs are all in the Flask backend:

| Title | Area |
|---|---|
| POST /tasks always assigns id=1 | ID counter never increments |
| DELETE /tasks/:id removes all tasks except the target | Filter condition inverted |
| GET /tasks/stats always reports count of 1 per status | Assignment instead of increment |
| PUT /tasks/:id does not update the updated_at timestamp | Missing timestamp refresh |
| GET /tasks?status= filter is case-sensitive | String comparison not lowercased |

## Step 9 — (Optional) Fix all remaining issues in one go

In [ ]:
remaining = json.loads(run(
    f"gh issue list --repo {GITHUB_USER}/{GITHUB_REPO} --state open "
    f"--label bug --json number,title --limit 20",
    capture=True
))

if not remaining:
    print("No open bug issues — all fixed!")
else:
    print(f"Fixing {len(remaining)} remaining bug issues sequentially...\n")
    for issue in remaining:
        n, title = issue["number"], issue["title"]
        print(f"\n{'='*60}\nIssue #{n}: {title}\n{'='*60}")
        prompt = (
            f"Fix GitHub issue #{n} in the repository {GITHUB_USER}/{GITHUB_REPO}. "
            f"Read the issue, locate the bug in the code, apply the fix, and open a pull request."
        )
        run(f'{sys.executable} connect.py --prompt "{prompt}"', cwd=AGENT_DIR)
    print("\nAll bug issues processed.")

## Step 10 — Summary

In [ ]:
prs = json.loads(run(
    f"gh pr list --repo {GITHUB_USER}/{GITHUB_REPO} --state open --json number,title,url --limit 20",
    capture=True
))
open_issues = json.loads(run(
    f"gh issue list --repo {GITHUB_USER}/{GITHUB_REPO} --state open --json number --limit 20",
    capture=True
))

print(f"PRs submitted:     {len(prs)}")
print(f"Issues still open: {len(open_issues)}")
print()
for pr in prs:
    print(f"  #{pr['number']:3d}  {pr['title']}")
    print(f"         {pr['url']}")

## Step 11 — (Optional) Interactive terminal on the microVM

Open a live shell inside the Claude Code container and interact with Claude directly.

Run in a **terminal** (not Jupyter):

```bash
cd coding_agents/claude-code
python connect.py
```

Try asking Claude to explain a file, investigate the repo, or write a new feature.

---
## Key Takeaways

- **AgentCore Runtime** — managed microVM, no servers, scales to zero when idle
- **AgentCore Gateway** — exposes GitHub as IAM-authenticated MCP tools; no API keys in the container
- **S3 Files mount** — shared filesystem lets you update the MCP proxy without rebuilding the image
- **CLAUDE.md** — the key control surface; shapes behaviour, enforces guardrails, sets conventions
- **Stateless sessions** — each `connect.py` call is a fresh microVM; idempotent branch/PR logic is essential